# Attribute filtering and hierarchy saliency

This notebook keeps three mathematically distinct operators separate:

- extinction filtering and raster cutoff-contour visualization on max/min-trees;
- the Cousty persistence hierarchical-watershed edge map;
- Xu shape-space saliency for a Tree of Shapes.

Complete primary references and implementation-specific generalizations are documented in [`docs/saliency.md`](../docs/saliency.md#primary-references-and-implementation-correspondence).
Direct attribute-threshold filtering is shown last for all four hierarchies.

## 1. Set up the environment

The notebook imports the installed package directly with `import mmcfilters`. Environment preparation is documented in `README.md` and is not performed in notebook cells.

In [ ]:
import mmcfilters
from pathlib import Path

import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["figure.figsize"] = (16, 8)


def load_grayscale(path):
    image = cv.imread(str(path), cv.IMREAD_GRAYSCALE)
    if image is None:
        raise FileNotFoundError(path)
    return np.ascontiguousarray(image, dtype=np.uint8)


## 2. Build morphological trees

The same image is represented by a max-tree, a min-tree, and two Tree-of-Shapes variants. The edge projections use the same 8-neighbour image graph.

In [ ]:
input_image = load_grayscale(Path("../dat/imgObjetos.png"))

(num_rows, num_cols) = input_image.shape

max_tree = mmcfilters.MorphologicalTreeFactory.createMaxTree(input_image, radius=1.5)
min_tree = mmcfilters.MorphologicalTreeFactory.createMinTree(input_image, radius=1.5)
tos = mmcfilters.MorphologicalTreeFactory.createTreeOfShapes(input_image, interpolation=mmcfilters.ToSInterpolation.SelfDual)
tos_4c8c = mmcfilters.MorphologicalTreeFactory.createTreeOfShapes(input_image, interpolation=mmcfilters.ToSInterpolation.Min4cMax8c)

## 3. Filter max/min-trees by extinction

`ExtinctionValues` requires a globally monotone altitude order and is therefore applied only to component trees. Extrema are selected by extinction strength; the Tree of Shapes is handled separately with Xu shaping.

In [ ]:
max_tree_filter = mmcfilters.AttributeFilters(max_tree)
min_tree_filter = mmcfilters.AttributeFilters(min_tree)
tos_filter = mmcfilters.AttributeFilters(tos)
tos_4c8c_filter = mmcfilters.AttributeFilters(tos_4c8c)

num_extrema_to_keep = 6
extinction_selection = mmcfilters.ExtinctionSelectionPolicy.byTopK(num_extrema_to_keep)
attribute_type = mmcfilters.Attribute.AREA

max_tree_attribute = mmcfilters.Attribute.computeSingleAttribute(max_tree, attribute_type)
min_tree_attribute = mmcfilters.Attribute.computeSingleAttribute(min_tree, attribute_type)
tos_attribute = mmcfilters.Attribute.computeSingleAttribute(tos, attribute_type)
tos_4c8c_attribute = mmcfilters.Attribute.computeSingleAttribute(tos_4c8c, attribute_type)

max_extinction = mmcfilters.ExtinctionValues(max_tree, max_tree_attribute)
min_extinction = mmcfilters.ExtinctionValues(min_tree, min_tree_attribute)
max_extinction_filtered = max_extinction.filtering(extinction_selection)
min_extinction_filtered = min_extinction.filtering(extinction_selection)

fig, axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)
for ax, title, image in zip(
    axes,
    ("input", "extinction filtering — max-tree", "extinction filtering — min-tree"),
    (input_image, max_extinction_filtered, min_extinction_filtered),
):
    ax.imshow(image, cmap="gray", vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis("off")
plt.show()

### 3.1 Display selected cutoff contours

`contourMap(...)` returns a raster of contours associated with selected extrema. It depends on the top-k policy and is not the formal saliency map $\Phi(H)$, which is defined on every graph edge.

In [ ]:
max_extinction_contours = max_extinction.contourMap(
    extinction_selection,
    mmcfilters.ExtinctionContourScorePolicy.RankScore,
)
min_extinction_contours = min_extinction.contourMap(
    extinction_selection,
    mmcfilters.ExtinctionContourScorePolicy.RankScore,
)

fig, axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)
for ax, title, image in zip(
    axes,
    ("input", "top-k contours — max-tree", "top-k contours — min-tree"),
    (input_image, max_extinction_contours, min_extinction_contours),
):
    ax.imshow(image, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.show()

### 3.2 Build the Cousty persistence map

`computeFormalSaliencyEdgeMap(ranked=True)` builds an altitude-ordered MST/BPTAO, assigns every binary merge the minimum of the two maximum descendant extinctions, and extends the persistence-weighted MST to the full graph. `sources`, `targets`, and `values` are the canonical result; displayed images are derived endpoint aggregations.

In [ ]:
formal_edge_maps = {
    "max-tree": max_extinction.computeFormalSaliencyEdgeMap(ranked=True),
    "min-tree": min_extinction.computeFormalSaliencyEdgeMap(ranked=True),
}
formal_rasters = {
    name: mmcfilters.HierarchySaliencyMapProjection.edgeMapToPixelImage(
        edge_map,
        mmcfilters.EdgeToPixelReducer.Max,
    )
    for name, edge_map in formal_edge_maps.items()
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)
axes[0].imshow(input_image, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("input")
axes[0].axis("off")
for ax, (name, raster) in zip(axes[1:], formal_rasters.items()):
    artist = ax.imshow(raster, cmap="gray")
    ax.set_title(f"derived visualization — {name}")
    ax.axis("off")
    fig.colorbar(artist, ax=ax, fraction=0.046, pad=0.04)
plt.show()

for name, edge_map in formal_edge_maps.items():
    values = np.asarray(edge_map["values"])
    print(
        f"{name}: {values.size} edges, "
        f"{np.unique(values).size} effective levels, "
        f"range=[{int(values.min())}, {int(values.max())}]"
    )

## 4. Compute Xu shape-space saliency

A Tree of Shapes does not satisfy the global altitude-order assumption of `ExtinctionValues`. `ShapeSpaceSaliency` treats circularity as an arbitrary attribute, computes extrema in the second shape-space hierarchy, and accumulates their extinctions by maximum on original contours. This differs from the Cousty hierarchical watershed.

In [ ]:
tos_circularity = mmcfilters.Attribute.computeSingleTopologyAttribute(
    tos,
    mmcfilters.Attribute.CIRCULARITY,
).astype(np.float32, copy=False)
tos_circularity = np.ascontiguousarray(tos_circularity)

tos_shape_space = mmcfilters.ShapeSpaceSaliency.compute(
    tos,
    attribute=tos_circularity,
    polarity=mmcfilters.ShapeSpaceExtremaPolarity.Maxima,
    radius=1.5,
)
tos_shape_edge_map = tos_shape_space["edgeMap"]
tos_shape_raster = mmcfilters.HierarchySaliencyMapProjection.edgeMapToPixelImage(
    tos_shape_edge_map,
    mmcfilters.EdgeToPixelReducer.Max,
)

fig, axes = plt.subplots(1, 2, figsize=(10, 5), constrained_layout=True)
axes[0].imshow(input_image, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("input")
axes[0].axis("off")
artist = axes[1].imshow(tos_shape_raster, cmap="gray")
axes[1].set_title("derived visualization — Xu shaping")
axes[1].axis("off")
fig.colorbar(artist, ax=axes[1], fraction=0.046, pad=0.04)
plt.show()

print(f"shape-space extrema: {len(tos_shape_space['extrema'])}")
print(f"image edges: {len(tos_shape_edge_map['values'])}")


## 5. Filter directly by an attribute threshold

This operation uses the area scale directly on all four trees. It computes neither extinction values nor saliency maps.

In [ ]:
threshold = 10000

max_tree_filtered_image = max_tree_filter.filteringByPruningMax(max_tree_attribute > threshold)
min_tree_filtered_image = min_tree_filter.filteringByPruningMax(min_tree_attribute > threshold)
tos_filtered_image = tos_filter.filteringByPruningMax(tos_attribute > threshold)
tos_4c8c_filtered_image = tos_4c8c_filter.filteringByPruningMax(tos_4c8c_attribute > threshold)

fig, axes = plt.subplots(1, 5, figsize=(20, 4), constrained_layout=True)
for ax, title, image in zip(
    axes,
    ("input", "max-tree", "min-tree", "self-dual ToS", "ToS 4c/8c"),
    (input_image, max_tree_filtered_image, min_tree_filtered_image, tos_filtered_image, tos_4c8c_filtered_image),
):
    ax.imshow(image, cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"area threshold — {title}" if title != "input" else title)
    ax.axis("off")
plt.show()

## 6. Numerical checks

The checks confirm output domains and the distinction between raster images and edge-indexed functions.

In [ ]:
assert max_extinction_contours.shape == input_image.shape
assert min_extinction_contours.shape == input_image.shape
assert max_extinction_filtered.shape == input_image.shape
assert min_extinction_filtered.shape == input_image.shape
assert tos_shape_raster.shape == input_image.shape

all_edge_maps = {**formal_edge_maps, "shape-space ToS": tos_shape_edge_map}
for name, edge_map in all_edge_maps.items():
    sources = np.asarray(edge_map["sources"])
    targets = np.asarray(edge_map["targets"])
    values = np.asarray(edge_map["values"])
    assert sources.ndim == targets.ndim == values.ndim == 1, name
    assert sources.size == targets.size == values.size, name
    assert np.all(np.isfinite(values)), name
    assert np.all(values >= 0), name

tos_live_nodes = np.asarray(tos.aliveNodeIds, dtype=np.int64)
assert np.all(np.isfinite(tos_circularity[tos_live_nodes]))
print("numerical checks: OK")
